# Step 3 (Baru): Ekstraksi Fitur Spektral (Band Power + Differential Entropy)

**Tujuan:** Melengkapi fitur konektivitas (GC, PDC) dengan fitur spektral klasik yang terbukti kuat
untuk klasifikasi emosi EEG pada dataset SEED (Zheng & Lu, 2015; Duan et al., 2013).

**Tahap ini terpisah dari tahap lain** (paralel dengan `01_granger_causality` dan `02_pdc`) — input dari
`00_preprocessing/output/preprocessed_eeg/`, output langsung berupa CSV tidy per trial
(`03_spectral_features/csv/cleaned_spectral_features.csv`) dengan skema kolom kunci yang sama
persis dengan `phase_1_data_structuring/csv/cleaned_graph_metrics_*.csv`, supaya bisa langsung
di-*join* di `phase_3_feature_engineering` tanpa perlu tahap agregasi tambahan.

**Fitur per channel per band (5 band, sama seperti PDC: delta, theta, alpha, beta, gamma):**
- **Band power** (integral Welch PSD di rentang band)
- **Differential Entropy (DE)** = `0.5 * log(2*pi*e*power)` — pendekatan standar (Duan et al. 2013)
  yang setara dengan variansi sinyal ter-bandpass pada band tsb (Parseval), tanpa perlu filtering eksplisit.

Total fitur: 62 channel × 5 band × 2 metrik = **620 kolom fitur** (+ 6 kolom kunci).

In [1]:
# ============================================================
# SEL INI: IMPORT & KONFIGURASI
# ------------------------------------------------------------
# Load config.py global (path, band frequency, subject-session mapping,
# label mapping) supaya konsisten dengan tahap 00-02 yang sudah ada.
# ============================================================
import os
import sys
import time
import numpy as np
import pandas as pd
from scipy.signal import welch

sys.path.insert(0, r"D:\Skripsi\new_data")
from config import (
    PREPROCESS_DIR, TARGET_FS, N_CHANNELS, CHANNEL_NAMES,
    SUBJECT_SESSIONS, TRIAL_LABELS, EMOTION_MAP, LABEL_MAP,
    PDC_FREQUENCY_BANDS, N_SUBJECTS,
)

OUTPUT_DIR = r"D:\Skripsi\new_data\03_spectral_features\csv"
FIG_DIR = r"D:\Skripsi\new_data\03_spectral_features\figures"
os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(FIG_DIR, exist_ok=True)

BANDS = PDC_FREQUENCY_BANDS  # {'delta': (0.5,4), 'theta': (4,8), 'alpha': (8,13), 'beta': (13,30), 'gamma': (30,45)}
N_TRIALS = 15

print(f"Target FS       : {TARGET_FS} Hz")
print(f"N channels      : {N_CHANNELS}")
print(f"Bands           : {BANDS}")
print(f"Preprocess dir  : {PREPROCESS_DIR}")
print(f"Output dir      : {OUTPUT_DIR}")

Target FS       : 100 Hz
N channels      : 62
Bands           : {'delta': (0.5, 4), 'theta': (4, 8), 'alpha': (8, 13), 'beta': (13, 30), 'gamma': (30, 45)}
Preprocess dir  : D:\Skripsi\new_data\00_preprocessing\output\preprocessed_eeg
Output dir      : D:\Skripsi\new_data\03_spectral_features\csv


In [2]:
# ============================================================
# SEL INI: FUNGSI EKSTRAKSI FITUR (band power + DE) PER TRIAL
# ------------------------------------------------------------
# Welch PSD dihitung SEKALI per channel (bukan per band) lalu
# di-integralkan (trapezoid) ke tiap rentang band -> jauh lebih
# cepat daripada bandpass-filter per band per channel.
#
# DE dihitung dari band power langsung (bukan filtfilt + varians):
# untuk sinyal zero-mean, varians sinyal ter-bandpass ~ integral PSD
# di band tsb (teorema Parseval) -> DE = 0.5*log(2*pi*e*power).
# Ini adalah pendekatan standar di literatur EEG-emotion (Duan et al. 2013).
# ============================================================
def extract_trial_features(eeg, fs, bands, channel_names):
    """eeg: (n_channels, timepoints) -> dict fitur spec_power_/spec_de_ per band per channel"""
    n_channels = eeg.shape[0]
    nperseg = min(eeg.shape[1], int(fs * 4))  # jendela 4 detik -> resolusi freq 0.25 Hz

    feats = {}
    for c in range(n_channels):
        ch_name = channel_names[c]
        freqs, psd = welch(eeg[c], fs=fs, nperseg=nperseg)
        for band_name, (lo, hi) in bands.items():
            mask = (freqs >= lo) & (freqs <= hi)
            power = np.trapz(psd[mask], freqs[mask]) if mask.sum() > 1 else 0.0
            power = max(power, 1e-12)
            de = 0.5 * np.log(2 * np.pi * np.e * power)
            feats[f"spec_power_{band_name}_{ch_name}"] = power
            feats[f"spec_de_{band_name}_{ch_name}"] = de
    return feats

print("Fungsi extract_trial_features siap.")

Fungsi extract_trial_features siap.


In [3]:
# ============================================================
# SEL INI: LOOP SEMUA SUBJECT-SESSION-TRIAL
# ------------------------------------------------------------
# Membaca .npy hasil 00_preprocessing, ekstrak fitur spektral,
# lalu tempelkan kolom kunci (subject_id, subject_num, session,
# trial, class, class_label) dengan format IDENTIK ke
# phase_1_data_structuring/csv/cleaned_graph_metrics_*.csv
# supaya bisa langsung di-merge di Phase 3.
# ============================================================
rows = []
t_start = time.time()
n_done, n_missing = 0, 0

for subject_num, sessions in sorted(SUBJECT_SESSIONS.items()):
    subject_id = f"S{subject_num:02d}"
    for session in sessions:
        session_dir = os.path.join(PREPROCESS_DIR, f"subject_{subject_num:02d}", f"session_{session}")
        for trial_idx in range(N_TRIALS):
            trial_num = trial_idx + 1
            fpath = os.path.join(session_dir, f"trial_{trial_num:02d}.npy")
            if not os.path.exists(fpath):
                n_missing += 1
                continue

            eeg = np.load(fpath)
            feats = extract_trial_features(eeg, TARGET_FS, BANDS, CHANNEL_NAMES)

            raw_label = TRIAL_LABELS[trial_idx]
            row = {
                "subject_id": subject_id,
                "subject_num": subject_num,
                "session": int(session),
                "trial": trial_num,
                "class": LABEL_MAP[raw_label],
                "class_label": EMOTION_MAP[raw_label],
            }
            row.update(feats)
            rows.append(row)
            n_done += 1

    elapsed = time.time() - t_start
    print(f"  Subject {subject_id}: selesai ({n_done} trial terkumpul, {elapsed:.1f}s)")

print(f"\nSelesai. Total trial diproses: {n_done}, hilang/tidak ditemukan: {n_missing}")
print(f"Waktu total: {time.time() - t_start:.1f}s")

C:\Users\aditp\AppData\Local\Temp\ipykernel_19272\4024635506.py:24: DeprecationWarning: `trapz` is deprecated. Use `trapezoid` instead, or one of the numerical integration functions in `scipy.integrate`.
  power = np.trapz(psd[mask], freqs[mask]) if mask.sum() > 1 else 0.0


  Subject S01: selesai (45 trial terkumpul, 26.9s)


  Subject S02: selesai (90 trial terkumpul, 52.0s)


  Subject S03: selesai (135 trial terkumpul, 71.2s)


  Subject S04: selesai (180 trial terkumpul, 82.3s)


...

  Subject S11: selesai (495 trial terkumpul, 260.5s)


  Subject S12: selesai (540 trial terkumpul, 283.1s)


  Subject S13: selesai (585 trial terkumpul, 295.3s)


  Subject S14: selesai (630 trial terkumpul, 307.0s)


  Subject S15: selesai (675 trial terkumpul, 318.1s)

Selesai. Total trial diproses: 675, hilang/tidak ditemukan: 0
Waktu total: 318.1s


In [4]:
# ============================================================
# SEL INI: SUSUN DATAFRAME & SIMPAN CSV
# ============================================================
df_spectral = pd.DataFrame(rows)
key_cols = ["subject_id", "subject_num", "session", "trial", "class", "class_label"]
feature_cols = [c for c in df_spectral.columns if c not in key_cols]
df_spectral = df_spectral[key_cols + feature_cols]

print(f"Shape akhir: {df_spectral.shape}")
print(f"Jumlah fitur: {len(feature_cols)} (harapan: 62 channel x 5 band x 2 metrik = 620)")

out_path = os.path.join(OUTPUT_DIR, "cleaned_spectral_features.csv")
df_spectral.to_csv(out_path, index=False)
print(f"Tersimpan: {out_path}")

df_spectral[key_cols].head(10)

Shape akhir: (675, 626)
Jumlah fitur: 620 (harapan: 62 channel x 5 band x 2 metrik = 620)


Tersimpan: D:\Skripsi\new_data\03_spectral_features\csv\cleaned_spectral_features.csv


,subject_id,subject_num,session,trial,class,class_label
0,S01,1,20131027,1,2,positive
1,S01,1,20131027,2,1,neutral
2,S01,1,20131027,3,0,negative
3,S01,1,20131027,4,0,negative
4,S01,1,20131027,5,1,neutral
5,S01,1,20131027,6,2,positive
6,S01,1,20131027,7,0,negative
7,S01,1,20131027,8,1,neutral
8,S01,1,20131027,9,2,positive
9,S01,1,20131027,10,2,positive


In [5]:
# ============================================================
# SEL INI: SANITY CHECK CEPAT
# ------------------------------------------------------------
# - Pastikan tidak ada NaN
# - Pastikan distribusi kelas seimbang
# - Quick F-score check (ANOVA) untuk melihat apakah fitur spektral
#   informatif terhadap label emosi (dibandingkan nanti dengan GC/PDC
#   di Phase 3 - 3.3_informativeness_comparison)
# ============================================================
from sklearn.feature_selection import f_classif

print("Missing values:", df_spectral[feature_cols].isna().sum().sum())
print("\nDistribusi kelas:")
print(df_spectral["class_label"].value_counts())

X = df_spectral[feature_cols].fillna(0).values
y = df_spectral["class"].values
f_scores, _ = f_classif(X, y)
print(f"\nMean F-score (ANOVA) semua fitur spektral: {np.nanmean(f_scores):.4f}")
print("Top 10 fitur paling diskriminatif:")
top_idx = np.argsort(-np.nan_to_num(f_scores))[:10]
for i in top_idx:
    print(f"  {feature_cols[i]:35s} F={f_scores[i]:.2f}")

Missing values: 0

Distribusi kelas:
class_label
positive    225
neutral     225
negative    225
Name: count, dtype: int64

Mean F-score (ANOVA) semua fitur spektral: 6.5160
Top 10 fitur paling diskriminatif:
  spec_de_gamma_T7                    F=139.36
  spec_de_beta_T7                     F=125.12
  spec_de_gamma_T8                    F=109.98
  spec_de_beta_T8                     F=102.48
  spec_de_gamma_FT8                   F=89.11
  spec_de_gamma_TP7                   F=84.06
  spec_de_beta_FT8                    F=73.39
  spec_de_gamma_FT7                   F=68.20
  spec_de_beta_TP7                    F=67.94
  spec_de_gamma_P8                    F=62.95
